# **Raw functions**

In [ ]:
import cv2
import numpy as np

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    d = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(d)]
    rect[3] = pts[np.argmax(d)]

    return rect

def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br-bl)
    widthB = np.linalg.norm(tr-tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr-br)
    heightB = np.linalg.norm(tl-bl)
    maxHeight = int(max(heightA, heightB))

    margin = 20
    
    dst = np.array([
        [margin, margin],
        [margin + maxWidth - 1, margin],
        [margin + maxWidth - 1, margin + maxHeight - 1],
        [margin, margin + maxHeight-1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth + 2*margin, maxHeight + 2*margin))
    return warped
    
def detect_walls(img):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #thresh = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 81, 31)
    _, thresh = cv2.threshold(img_gray, 60, 255, cv2.THRESH_BINARY_INV)
    
    analysis = cv2.connectedComponentsWithStats(thresh, 4, cv2.CV_32S)
    (totalLabels, label_ids, values, centroid) = analysis
    
    walls = np.zeros(thresh.shape, dtype="uint8")
    min_area = 6000
    
    for i in range(1, totalLabels):
        area = values[i, cv2.CC_STAT_AREA]  

        if (area > min_area) :
            componentMask = (label_ids == i).astype("uint8") * 255
            walls = cv2.bitwise_or(walls, componentMask)
            
    return walls
    
def add_safety_margin(grid, margin=3):
    wall_mask = (grid == 1).astype(np.uint8)

    size = 2*margin + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
    dilated = cv2.dilate(wall_mask, kernel)

    grid[(dilated == 1) & (grid == 0)] = 4
    return grid

def to_grid_walls(walls, margin, rows, cols):
    grid = overlay = np.zeros((rows, cols), dtype=np.uint8)
    h, w = walls.shape
    
    for r in range(rows):
        for c in range(cols):
            y_start = int(r*h/rows)
            y_end   = int((r + 1)*h/rows)
            x_start = int(c*w/cols)
            x_end   = int((c + 1)*w/cols)
            
            cell = walls[y_start:y_end, x_start:x_end]
            if np.mean(cell) > 100:
                grid[r, c] = 1 
            else:
                grid[r, c] = 0

    grid = add_safety_margin(grid, margin=margin)
    
    return grid

def to_grid_pos(grid, thymio_coord, finish_coord, rows, cols, h, w):
    if thymio_coord != None:
        x_thymio, y_thymio = thymio_coord[0]
        
        c = int(x_thymio*cols/w)
        r = int(y_thymio*rows/h)
        c_end = int(c + 5*np.cos(thymio_coord[1]))
        r_end = int(r + 5*np.sin(thymio_coord[1]))
        
        grid[r, c] = 2
        grid = cv2.circle(grid, (c, r), 5, 2, 1)
        grid = cv2.line(grid, (c, r), (c_end, r_end), 2, 1)

    if finish_coord != None:
        x_finish, y_finish = finish_coord
        r = int(y_finish*rows/h)
        c = int(x_finish*cols/w)
        grid[r, c] = 3
    
    return grid

def draw_grid(grid):
    rows, cols = grid.shape
    img = np.full((rows, cols, 3), 255, dtype=np.uint8)   # white background (BGR)

    img[grid == 1] = (0, 0, 0)
    img[grid == 2] = (0, 200, 0)
    img[grid == 3] = (0, 0, 255)
    img[grid == 4] = (180, 180, 180)

    return img
    
def find_light(mask, min_area=20):
    contour, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contour:
        return None

    largest = max(contour, key=cv2.contourArea)
    
    if cv2.contourArea(largest) < min_area:
        return None
    
    M = cv2.moments(largest)

    if M['m00'] == 0:
        return None
    
    cx = int(M['m10']/M['m00'])
    cy = int(M['m01']/M['m00'])

    return cx, cy

def detect_finish(img):
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    lower1 = np.array([0, 120, 70])
    upper1 = np.array([10, 255, 255])

    lower2 = np.array([170, 120, 70])
    upper2 = np.array([179, 255, 255])

    mask1 = cv2.inRange(img_hsv, lower1, upper1)
    mask2 = cv2.inRange(img_hsv, lower2, upper2)
    mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    finish_coord = find_light(mask)

    if finish_coord == None:
        return None

    return finish_coord

def detect_thymio(img):
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_g = np.array([35, 80, 80])
    upper_g = np.array([85, 255, 255])
    lower_b = np.array([90, 80, 80])
    upper_b = np.array([120, 255, 255])
    
    mask_g = cv2.inRange(hsv, lower_g, upper_g)
    mask_b = cv2.inRange(hsv, lower_b, upper_b)
    
    kernel = np.ones((5, 5), np.uint8)
    
    mask_g = cv2.morphologyEx(mask_g, cv2.MORPH_OPEN, kernel)
    mask_b = cv2.morphologyEx(mask_b, cv2.MORPH_OPEN, kernel)

    light_g = find_light(mask_g)
    light_b = find_light(mask_b)
    
    if light_g == None or light_b == None:
        return None

    cx_g, cy_g = light_g
    cx_b, cy_b = light_b

    center_g = [cx_g, cy_g]
    center_b = [cx_b, cy_b]
    
    theta = np.arctan2(cy_g - cy_b, cx_g - cx_b)
    center = [(cx_g + cx_b)//2, (cy_g + cy_b)//2]
    
    return center, theta

def draw_direction(img, coordinates):
    if coordinates == None :
        return img, None

    center, theta = coordinates
    
    img_full = cv2.circle(img, center, 5, (0, 0, 255), -1)

    x_end = int(center[0] + 100*np.cos(theta))
    y_end = int(center[1] + 100*np.sin(theta))

    img_full = cv2.arrowedLine(img_full, center, (x_end, y_end), (100, 100, 0), 5)

    sz = img.shape
    h, w = sz[0], sz[1]
    
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    cv2.circle(overlay, center, 5, (0, 0, 255, 255), -1)
    cv2.arrowedLine(overlay, center, (x_end, y_end), (100, 100, 0, 255), 5)
    
    return img_full, overlay

def draw_finish(img, coordinates):
    if coordinates == None:
        return None
    
    sz = img.shape
    h, w = sz[0], sz[1]
    
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    cv2.circle(overlay, coordinates, 5, (0, 0, 255, 255), -1)

    return overlay

def merge(base_bgr, overlay_bgra):
    overlay_bgra = cv2.resize(overlay_bgra, (base_bgr.shape[1], base_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    alpha = overlay_bgra[:, :, 3:4] / 255.0 
    out = base_bgr * (1 - alpha) + overlay_bgra[:, :, :3] * alpha
    return out.astype(np.uint8)

def capture_obstacles(sec, cap, rows, cols):
    for i in range(sec*10):
        ok, frame = cap.read()
        _, jpeg = cv2.imencode('.jpg', frame)
        ecran.value = jpeg.tobytes()
        
    ok, frame = cap.read()
    
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(frame, pts.astype("float32"))
            walls = detect_walls(warped)

            h, w = walls.shape
            
            grid = to_grid_walls(walls, 15, rows, cols)
    
    return grid, h, w

# **Run code**

**Run code**

In [2]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

In [ ]:
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

ecran = widgets.Image(format="jpeg")
display(ecran)

rows = 160
cols = 200

obstacles_grid, h, w = capture_obstacles(5, cap, rows, cols)
    
while True:
    ok, frame = cap.read()
    if not ok:
        break
        
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(frame, pts.astype("float32"))
            
            finish_coord = detect_finish(warped)
            thymio_coord = detect_thymio(warped)
            
            grid_full = to_grid_pos(obstacles_grid.copy(), thymio_coord, finish_coord, rows, cols, h, w)
            grid_img = draw_grid(grid_full)
            grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)
            
            _, jpeg = cv2.imencode('.jpg', grid_img)
            ecran.value = jpeg.tobytes()
            
            continue
            
    _, jpeg = cv2.imencode('.jpg', frame)
    ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

TypeError: capture_obstacles() missing 1 required positional argument: 'margin'